In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2017_2018"
    / "Examination"
)

xpt_file = DATA_DIR / "BMX_J.xpt"
csv_file = DATA_DIR / "BMX_J_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve completely untouched pandas-decoded source.
#
source_df = df.copy()


print("=" * 90)
print("NHANES 2017-2018 BMX_J XPT -> CSV")
print("=" * 90)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "BMDSTATS",
    "BMXWT",
    "BMIWT",
    "BMXRECUM",
    "BMIRECUM",
    "BMXHEAD",
    "BMIHEAD",
    "BMXHT",
    "BMIHT",
    "BMXBMI",
    "BMXLEG",
    "BMILEG",
    "BMXARML",
    "BMIARML",
    "BMXARMC",
    "BMIARMC",
    "BMXWAIST",
    "BMIWAIST",
    "BMXHIP",
    "BMIHIP"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "BMX_J column names/order differ from expected.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 8704:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 21:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 8,704 rows x 21 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# WHOLE-NUMBER / CODE FIELDS
# ------------------------------------------------------------
#
# SEQN
#     respondent identifier
#
# BMDSTATS
#     body-measures component status
#
# BMIxxxx fields
#     measurement comment/status codes
#
#
# GENUINE DECIMAL MEASUREMENTS
# ------------------------------------------------------------
#
# BMXWT
#     Weight kg
#
# BMXRECUM
#     Recumbent length cm
#
# BMXHEAD
#     Head circumference cm
#
# BMXHT
#     Standing height cm
#
# BMXBMI
#     BMI kg/m^2
#
# BMXLEG
#     Upper leg length cm
#
# BMXARML
#     Upper arm length cm
#
# BMXARMC
#     Arm circumference cm
#
# BMXWAIST
#     Waist circumference cm
#
# BMXHIP
#     Hip circumference cm
#
#
# IMPORTANT:
#
# Measurement variables remain decimal even when an
# individual value happens to equal something such as:
#
#     55.0
#     40.0
#     179.0
#
# The VARIABLE is still a decimal measurement.
#

INTEGER_COLUMNS = [
    "SEQN",
    "BMDSTATS",
    "BMIWT",
    "BMIRECUM",
    "BMIHEAD",
    "BMIHT",
    "BMILEG",
    "BMIARML",
    "BMIARMC",
    "BMIWAIST",
    "BMIHIP"
]


DECIMAL_COLUMNS = [
    "BMXWT",
    "BMXRECUM",
    "BMXHEAD",
    "BMXHT",
    "BMXBMI",
    "BMXLEG",
    "BMXARML",
    "BMXARMC",
    "BMXWAIST",
    "BMXHIP"
]


if set(
    INTEGER_COLUMNS
    +
    DECIMAL_COLUMNS
) != set(EXPECTED_COLUMNS):

    raise ValueError(
        "Field type plan does not cover "
        "all BMX_J variables."
    )


print("\n--- FIELD TYPE PLAN ---")

print(
    "Whole-number/code fields:",
    INTEGER_COLUMNS
)

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)


# ============================================================
# 5. VERIFY ALL FIELDS LOADED NUMERICALLY
# ============================================================

numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all BMX_J variables to load "
        "numerically.\n\n"
        f"Actual numeric columns:\n{numeric_cols}"
    )


print(
    "PASS: All 21 variables loaded numerically."
)


# ============================================================
# 6. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known SAS XPORT / pandas decoded-zero artifact:
#
#     5.397605346934028e-79
#
# Direct inspection of this BMX_J attachment:
#
#     expected tiny count = 0
#
# Therefore NO zero correction should be made.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


tiny_counts = (
    df[EXPECTED_COLUMNS]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


print(
    "\n--- ORIGINAL XPT TINY-VALUE CHECK ---"
)

print(
    tiny_counts.to_string()
)

print(
    "Total tiny XPORT artifacts:",
    tiny_total
)


if tiny_total != 0:

    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "found in BMX_J.\n\n"
        + tiny_counts[
            tiny_counts > 0
        ].to_string()
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 7. ORIGINAL ZERO CHECK
# ============================================================

zero_counts = (
    df[EXPECTED_COLUMNS]
    .eq(0)
    .sum()
)


zero_total = int(
    zero_counts.sum()
)


print(
    "\n--- ORIGINAL NUMERIC ZERO CHECK ---"
)

print(
    zero_counts.to_string()
)

print(
    "Total ordinary numeric zeros:",
    zero_total
)


if zero_total != 0:

    raise ValueError(
        "Unexpected numeric zeros found "
        "in attached BMX_J."
    )


print(
    "PASS: Original BMX_J contains no numeric zeros."
)


# ============================================================
# 8. EXPECTED CORRECTED SOURCE
# ============================================================

#
# BMX_J requires NO numeric correction.
#
# Therefore:
#
#     expected_df == source_df
#

expected_df = source_df.copy()


# ============================================================
# 9. FRACTIONAL VALUE DISCOVERY
# ============================================================

#
# Run BEFORE any Int64 conversion.
#
# np.round() is used ONLY for comparison.
#
# It DOES NOT modify the dataframe.
#

print(
    "\n--- FRACTIONAL VALUE DISCOVERY ---"
)


fractional_counts = {}


for col in EXPECTED_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        fractional_count = 0

    else:

        fractional_mask = ~np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )

        fractional_count = int(
            fractional_mask.sum()
        )


    fractional_counts[col] = fractional_count


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


EXPECTED_FRACTIONAL_COUNTS = {

    "SEQN": 0,

    "BMDSTATS": 0,

    "BMXWT": 7738,
    "BMIWT": 0,

    "BMXRECUM": 801,
    "BMIRECUM": 0,

    "BMXHEAD": 182,
    "BMIHEAD": 0,

    "BMXHT": 7181,
    "BMIHT": 0,

    "BMXBMI": 7190,

    "BMXLEG": 5219,
    "BMILEG": 0,

    "BMXARML": 5767,
    "BMIARML": 0,

    "BMXARMC": 6904,
    "BMIARMC": 0,

    "BMXWAIST": 6505,
    "BMIWAIST": 0,

    "BMXHIP": 5118,
    "BMIHIP": 0
}


if fractional_counts != EXPECTED_FRACTIONAL_COUNTS:

    raise ValueError(
        "Unexpected BMX_J fractional-value pattern.\n\n"
        f"Expected:\n{EXPECTED_FRACTIONAL_COUNTS}\n\n"
        f"Actual:\n{fractional_counts}"
    )


print(
    "PASS: Decimal and whole-number variables "
    "identified correctly."
)


# ============================================================
# 10. PROVE INTEGER/CODE FIELDS ARE SAFE
# ============================================================

print(
    "\n--- INTEGER/CODE FIELD VALIDATION ---"
)


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        print(
            f"{col}: all missing"
        )

        continue


    whole_mask = np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        (~whole_mask)
        .sum()
    )


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count != 0:

        raise ValueError(
            f"{col} unexpectedly contains "
            "fractional values."
        )


print(
    "PASS: All ID/status/comment fields "
    "are whole-number."
)


# ============================================================
# 11. VALIDATE SEQN
# ============================================================

print(
    "\n--- SEQN ---"
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 93703:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 102956:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains duplicate values."
    )


print(
    "PASS: SEQN 93703-102956 validated."
)


# ============================================================
# 12. FREQUENCY CHECK HELPER
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    print(
        f"\n--- CDC CHECK: {column} ---"
    )

    print(
        "Observed:",
        actual_values
    )

    print(
        "Missing:",
        actual_missing
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ from CDC.\n\n"
            f"Expected:\n{expected_values}\n\n"
            f"Actual:\n{actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 13. COMPONENT STATUS CODE
# ============================================================

check_frequency(
    "BMDSTATS",
    {
        1: 8019,
        2: 295,
        3: 283,
        4: 107
    },
    0
)


# ============================================================
# 14. COMMENT CODE CHECKS
# ============================================================

check_frequency(
    "BMIWT",
    {
        1: 11,
        3: 382,
        4: 23
    },
    8288
)


check_frequency(
    "BMIRECUM",
    {
        1: 24
    },
    8680
)


check_frequency(
    "BMIHEAD",
    {},
    8704
)


check_frequency(
    "BMIHT",
    {
        1: 32,
        3: 67
    },
    8605
)


check_frequency(
    "BMILEG",
    {
        1: 334
    },
    8370
)


check_frequency(
    "BMIARML",
    {
        1: 347
    },
    8357
)


check_frequency(
    "BMIARMC",
    {
        1: 350
    },
    8354
)


check_frequency(
    "BMIWAIST",
    {
        1: 437
    },
    8267
)


check_frequency(
    "BMIHIP",
    {
        1: 270
    },
    8434
)


# ============================================================
# 15. MEASUREMENT RANGE CHECK HELPER
# ============================================================

def check_measurement(
    column,
    expected_nonmissing,
    expected_missing,
    expected_min,
    expected_max
):

    nonmissing = int(
        df[column]
        .notna()
        .sum()
    )


    missing = int(
        df[column]
        .isna()
        .sum()
    )


    minimum = float(
        df[column]
        .min()
    )


    maximum = float(
        df[column]
        .max()
    )


    print(
        f"\n--- CDC MEASUREMENT CHECK: {column} ---"
    )

    print(
        "Nonmissing:",
        nonmissing
    )

    print(
        "Missing:",
        missing
    )

    print(
        "Minimum:",
        minimum
    )

    print(
        "Maximum:",
        maximum
    )


    if nonmissing != expected_nonmissing:

        raise ValueError(
            f"{column} nonmissing count differs.\n"
            f"Expected: {expected_nonmissing:,}\n"
            f"Actual:   {nonmissing:,}"
        )


    if missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {missing:,}"
        )


    if not np.isclose(
        minimum,
        expected_min,
        rtol=0,
        atol=0
    ):

        raise ValueError(
            f"{column} minimum differs."
        )


    if not np.isclose(
        maximum,
        expected_max,
        rtol=0,
        atol=0
    ):

        raise ValueError(
            f"{column} maximum differs."
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 16. WEIGHT
# ============================================================

check_measurement(
    "BMXWT",
    8580,
    124,
    3.2,
    242.6
)


# ============================================================
# 17. RECUMBENT LENGTH
# ============================================================

check_measurement(
    "BMXRECUM",
    894,
    7810,
    49.1,
    111.3
)


# ============================================================
# 18. HEAD CIRCUMFERENCE
# ============================================================

check_measurement(
    "BMXHEAD",
    194,
    8510,
    32.4,
    47.9
)


# ============================================================
# 19. STANDING HEIGHT
# ============================================================

check_measurement(
    "BMXHT",
    8016,
    688,
    78.3,
    197.7
)


# ============================================================
# 20. BODY MASS INDEX
# ============================================================

check_measurement(
    "BMXBMI",
    8005,
    699,
    12.3,
    86.2
)


# ============================================================
# 21. UPPER LEG LENGTH
# ============================================================

check_measurement(
    "BMXLEG",
    6703,
    2001,
    24.8,
    55.0
)


# ============================================================
# 22. UPPER ARM LENGTH
# ============================================================

check_measurement(
    "BMXARML",
    8177,
    527,
    9.4,
    49.9
)


# ============================================================
# 23. ARM CIRCUMFERENCE
# ============================================================

check_measurement(
    "BMXARMC",
    8173,
    531,
    11.2,
    56.3
)


# ============================================================
# 24. WAIST CIRCUMFERENCE
# ============================================================

check_measurement(
    "BMXWAIST",
    7601,
    1103,
    40.0,
    169.5
)


# ============================================================
# 25. HIP CIRCUMFERENCE
# ============================================================

check_measurement(
    "BMXHIP",
    6039,
    2665,
    66.6,
    179.0
)


# ============================================================
# 26. DECIMAL GRID CHECK
# ============================================================

#
# All released BMX measurement fields are on a
# one-decimal-place measurement grid.
#
# IMPORTANT:
#
# This is a VALIDATION ONLY.
#
# Nothing is rounded.
#
# The original source decimals remain untouched.
#

print(
    "\n--- DECIMAL 0.1 GRID CHECK ---"
)


decimal_grid_failures = {}


for col in DECIMAL_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    grid_mask = np.isclose(
        values * 10,
        np.round(
            values * 10
        ),
        rtol=0,
        atol=1e-10
    )


    failure_count = int(
        (~grid_mask)
        .sum()
    )


    print(
        f"{col}: "
        f"{failure_count:,} off-grid values"
    )


    if failure_count > 0:

        decimal_grid_failures[col] = failure_count


if decimal_grid_failures:

    raise ValueError(
        "Unexpected decimal precision found:\n"
        + str(decimal_grid_failures)
    )


print(
    "PASS: All body measurements are on "
    "the expected 0.1-unit grid."
)


# ============================================================
# 27. VERIFY DECIMAL VARIABLES UNCHANGED
# ============================================================

#
# No repairs have been performed.
#
# Every measurement variable must remain bit-for-bit
# numerically equal to the source float.
#

print(
    "\n--- DECIMAL PRE-CONVERSION FIDELITY ---"
)


decimal_failures = []


for col in DECIMAL_COLUMNS:

    original_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        original_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        decimal_failures.append(
            col
        )


if decimal_failures:

    raise ValueError(
        "Decimal measurement variables changed:\n"
        + str(decimal_failures)
    )


print(
    "PASS: All 10 genuine decimal measurement "
    "variables remain unchanged."
)


# ============================================================
# 28. CONVERT ONLY PROVEN INTEGER/CODE FIELDS
# ============================================================

#
# Convert:
#
#     SEQN
#     BMDSTATS
#     BMI comment-code variables
#
# to nullable Int64.
#
#
# DO NOT TOUCH BMX measurement variables.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} unexpectedly contains "
                "fractional values."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: ID/status/comment fields converted "
    "to nullable Int64."
)

print(
    "PASS: Anthropometric measurements retained "
    "as genuine decimals."
)


# ============================================================
# 29. COMPLETE PRE-EXPORT NUMERIC FIDELITY
# ============================================================

#
# BMX_J permitted numeric changes:
#
#     NONE
#
# Int64 conversion changes display representation only.
#

print(
    "\n--- COMPLETE PRE-EXPORT NUMERIC FIDELITY ---"
)


pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(pre_export_failures)
    )


print(
    "PASS: All 21 BMX_J variables retain "
    "their exact source numeric values."
)


# ============================================================
# 30. MISSING-POSITION FIDELITY BEFORE EXPORT
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    source_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )


    current_missing = (
        df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            source_missing
            !=
            current_missing
        )
    )


print(
    "\nMissing-position mismatches before export:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 31. FINAL ZERO PATTERN BEFORE EXPORT
# ============================================================

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if count > 0:

        final_zero_counts[col] = count


print(
    "\n--- FINAL ZERO PATTERN ---"
)

print(
    final_zero_counts
)


if final_zero_counts != {}:

    raise ValueError(
        "Unexpected numeric zeros introduced:\n"
        + str(final_zero_counts)
    )


print(
    "PASS: No numeric zeros introduced."
)


# ============================================================
# 32. EXPORT CSV
# ============================================================

#
# CRITICAL FIDELITY POLICY
# ------------------------------------------------------------
#
# DO NOT:
#
#     df.round(...)
#
# DO NOT:
#
#     float_format=
#
# DO NOT:
#
#     convert decimal BMX fields to integer
#
# DO NOT:
#
#     force one decimal place through formatting
#
#
# We preserve the actual numeric values decoded from
# the XPT.
#
# This approach also protects genuine decimals in:
#
#     Dietary
#     Laboratory
#     Examination
#     survey weights
#     nutrients
#     environmental measurements
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 33. RE-READ WITH ROUND-TRIP FLOAT PARSER
# ============================================================

#
# Exact floating-point fidelity should use:
#
#     float_precision="round_trip"
#
# rather than reparsing raw strings with pd.to_numeric().
#

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV column order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 34. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT XPT -> CSV NUMERIC FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV numeric fidelity differences:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 21 BMX_J variables "
    "round-trip exactly."
)


# ============================================================
# 35. CSV MISSING-VALUE FIDELITY
# ============================================================

csv_missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    csv_missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\n--- CSV MISSING-VALUE FIDELITY ---"
)

print(
    "Missing-position mismatches:",
    csv_missing_mismatches
)


if csv_missing_mismatches != 0:

    raise ValueError(
        "Missing positions changed during CSV export."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 36. READ FINAL CSV AS RAW TEXT
# ============================================================

#
# Raw text is used for FORMAT checks only.
#
# It is NOT used for exact binary-float equality.
#

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


if raw_csv_df.shape != (
    8704,
    21
):

    raise ValueError(
        "Raw CSV dimensions changed."
    )


if (
    raw_csv_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "Raw CSV column order changed."
    )


# ============================================================
# 37. INTEGER/CODE FIELD .0 CHECK
# ============================================================

#
# ONLY ID/status/comment fields are checked.
#
# DO NOT apply this check to:
#
#     BMXWT
#     BMXHT
#     BMXBMI
#     BMXWAIST
#     etc.
#
# Those are genuine decimal measurement variables.
#

print(
    "\n--- INTEGER/CODE FIELD .0 CHECK ---"
)


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting in "
        "ID/status/comment fields:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary '.0' in "
    "integer/code fields."
)


# ============================================================
# 38. DECIMAL FIELD RAW-TEXT CHECK
# ============================================================

#
# Decimal BMX variables are NOT subjected to a ".0"
# removal test.
#
# Values such as:
#
#     BMXLEG = 55.0
#     BMXWAIST = 40.0
#     BMXHIP = 179.0
#
# are legitimate observations in decimal measurement
# variables.
#

print(
    "\n--- DECIMAL MEASUREMENT CSV CHECK ---"
)


for col in DECIMAL_COLUMNS:

    populated_count = int(
        raw_csv_df[col]
        .ne("")
        .sum()
    )


    dot_zero_count = int(
        (
            raw_csv_df[col]
            .ne("")
            &
            raw_csv_df[col]
            .str.endswith(".0")
        )
        .sum()
    )


    print(
        f"{col}:"
    )

    print(
        "  populated:",
        populated_count
    )

    print(
        "  tokens ending '.0' "
        "(allowed):",
        dot_zero_count
    )


print(
    "PASS: Decimal measurements retained "
    "without forced formatting."
)


# ============================================================
# 39. FINAL DECIMAL-FIELD EXACT FIDELITY
# ============================================================

#
# Use roundtrip_df for exact float comparisons.
#
# Do NOT use raw CSV -> pd.to_numeric() for exact
# binary-float equality.
#

print(
    "\n--- FINAL DECIMAL-FIELD EXACT FIDELITY ---"
)


decimal_csv_failures = []


for col in DECIMAL_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    final_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        final_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    print(
        f"{col}:",
        "PASS" if same else "FAIL"
    )


    if not same:

        decimal_csv_failures.append(
            col
        )


if decimal_csv_failures:

    raise ValueError(
        "Decimal fidelity failures:\n"
        + str(decimal_csv_failures)
    )


print(
    "PASS: Every anthropometric decimal field "
    "round-trips exactly."
)


# ============================================================
# 40. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values.eq(TINY_VALUE)
        .sum()
    )


print(
    "\n--- FINAL TINY-VALUE CHECK ---"
)

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts found in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 41. FINAL ZERO CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    count = int(
        values.eq(0)
        .sum()
    )


    if count > 0:

        csv_zero_counts[col] = count


print(
    "\n--- FINAL CSV ZERO PATTERN ---"
)

print(
    csv_zero_counts
)


if csv_zero_counts != {}:

    raise ValueError(
        "Unexpected numeric zeros in final CSV:\n"
        + str(csv_zero_counts)
    )


print(
    "PASS: Final CSV contains no numeric zeros."
)


# ============================================================
# 42. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_errors = {}


for col in EXPECTED_COLUMNS:

    tokens = raw_csv_df[col]


    count = int(
        tokens.str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


    if count > 0:

        scientific_errors[col] = count


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)

print(
    scientific_errors
)


if scientific_errors:

    raise ValueError(
        "Unexpected scientific notation found:\n"
        + str(scientific_errors)
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 43. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 93703:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 102956:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn.isna()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains missing SEQN."
    )


if int(
    final_seqn.duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final SEQN 93703-102956 valid "
    "and contains no '.0'."
)


# ============================================================
# 44. FINAL ALL-COLUMN SOURCE -> CSV FIDELITY
# ============================================================

print(
    "\n--- FINAL ALL-COLUMN SOURCE -> CSV FIDELITY ---"
)


final_fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    final_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        final_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        final_fidelity_failures.append(
            col
        )


if final_fidelity_failures:

    raise ValueError(
        "Final source -> CSV fidelity failures:\n"
        + str(final_fidelity_failures)
    )


print(
    "PASS: Final CSV preserves all original "
    "BMX_J numeric values exactly."
)


# ============================================================
# 45. ROUND-TRIP MAXIMUM DIFFERENCE DIAGNOSTIC
# ============================================================

#
# Expected:
#
#     0.0
#
# for every field.
#

print(
    "\n--- ROUND-TRIP MAXIMUM ABSOLUTE DIFFERENCES ---"
)


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    final_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    valid_mask = (
        ~np.isnan(expected_values)
        &
        ~np.isnan(final_values)
    )


    if valid_mask.any():

        max_difference = float(
            np.max(
                np.abs(
                    expected_values[valid_mask]
                    -
                    final_values[valid_mask]
                )
            )
        )

    else:

        max_difference = 0.0


    print(
        f"{col}: {max_difference}"
    )


# ============================================================
# 46. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 90
)

print(
    "BMX_J CONVERSION COMPLETE"
)

print(
    "=" * 90
)

print(
    "Cycle: 2017-2018"
)

print(
    "Component: Examination - Body Measures"
)

print(
    "Eligible population: All survey participants"
)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)

print(
    "Original tiny XPORT artifacts:",
    tiny_total
)

print(
    "Numeric zero restorations:",
    0
)

print(
    "Other numeric corrections:",
    0
)

print(
    "Whole-number/code fields:",
    len(INTEGER_COLUMNS)
)

print(
    "Genuine decimal measurement fields:",
    len(DECIMAL_COLUMNS)
)

print(
    "BMXWT range:",
    f"{df['BMXWT'].min()}-"
    f"{df['BMXWT'].max()}"
)

print(
    "BMXHT range:",
    f"{df['BMXHT'].min()}-"
    f"{df['BMXHT'].max()}"
)

print(
    "BMXBMI range:",
    f"{df['BMXBMI'].min()}-"
    f"{df['BMXBMI'].max()}"
)

print(
    "BMXWAIST range:",
    f"{df['BMXWAIST'].min()}-"
    f"{df['BMXWAIST'].max()}"
)

print(
    "BMXHIP range:",
    f"{df['BMXHIP'].min()}-"
    f"{df['BMXHIP'].max()}"
)

print(
    "Exact decimal round-trip failures: "
    "0 expected"
)

print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)

print(
    "Missing-position differences: "
    "0 expected"
)

print(
    "Tiny values remaining: "
    "0 expected"
)

print(
    "Unnecessary '.0' in ID/code fields: "
    "0 expected"
)

print(
    "Scientific notation: "
    "0 expected"
)

print(
    "=" * 90
)

NHANES 2017-2018 BMX_J XPT -> CSV
Rows: 8,704
Columns: 21
PASS: Structure = 8,704 rows x 21 columns.

--- FIELD TYPE PLAN ---
Whole-number/code fields: ['SEQN', 'BMDSTATS', 'BMIWT', 'BMIRECUM', 'BMIHEAD', 'BMIHT', 'BMILEG', 'BMIARML', 'BMIARMC', 'BMIWAIST', 'BMIHIP']
Genuine decimal fields: ['BMXWT', 'BMXRECUM', 'BMXHEAD', 'BMXHT', 'BMXBMI', 'BMXLEG', 'BMXARML', 'BMXARMC', 'BMXWAIST', 'BMXHIP']
PASS: All 21 variables loaded numerically.

--- ORIGINAL XPT TINY-VALUE CHECK ---
SEQN        0
BMDSTATS    0
BMXWT       0
BMIWT       0
BMXRECUM    0
BMIRECUM    0
BMXHEAD     0
BMIHEAD     0
BMXHT       0
BMIHT       0
BMXBMI      0
BMXLEG      0
BMILEG      0
BMXARML     0
BMIARML     0
BMXARMC     0
BMIARMC     0
BMXWAIST    0
BMIWAIST    0
BMXHIP      0
BMIHIP      0
Total tiny XPORT artifacts: 0
PASS: No tiny XPORT artifacts.

--- ORIGINAL NUMERIC ZERO CHECK ---
SEQN        0
BMDSTATS    0
BMXWT       0
BMIWT       0
BMXRECUM    0
BMIRECUM    0
BMXHEAD     0
BMIHEAD     0
BMXHT       0
BM